# 75 - Experimento MPNet + HDBSCAN

Este notebook agrupa descrições normalizadas de itens de supermercado usando embeddings do modelo `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` e HDBSCAN. A análise usa distância cosseno, não usa NCM nem rótulos humanos e trata ruído como resultado explícito do algoritmo. A amostra configurável mantém o custo do experimento controlado. A execução requer `sentence-transformers` no ambiente e pode baixar o modelo do Hugging Face no primeiro uso. Execute de cima para baixo em uma sessão limpa.


In [1]:
from datetime import datetime, timezone
from pathlib import Path
import logging
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook75")
pd.set_option("display.max_colwidth", 140)


## 1. Parâmetros

`min_cluster_size` define o menor grupo que o HDBSCAN pode selecionar; `min_samples` controla o quão conservadora é a estimativa de densidade. Quando `min_samples=None`, o algoritmo usa `min_cluster_size`. A seleção `eom` procura grupos persistentes, enquanto `leaf` tende a produzir grupos menores e mais específicos. A grade é exploratória; silhouette e cobertura não medem acurácia semântica.

Cada semente seleciona uma amostra reprodutível. O modelo MPNet gera uma representação semântica por descrição e esses embeddings são reutilizados em toda a grade de HDBSCAN daquela semente.

O HDBSCAN do scikit-learn não aceita pesos por observação. Cada descrição única contribui uma vez para o agrupamento; a frequência permanece disponível para ordenar clusters e descrições nos relatórios. A silhouette é calculada em uma amostra reprodutível dos itens não classificados como ruído para limitar o custo.


In [2]:
DATASET_NAME = "2025_NCMs_filtrados-sem-duplicatas"  # Nome da pasta em data/processed; altere apenas este valor para trocar a base.

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Não encontrei a raiz do repositório com data/processed/.")
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME
DATASET_FILES = sorted(DATASET_DIR.glob("*.parquet"))
if not DATASET_FILES:
    raise FileNotFoundError(f"Nenhum Parquet encontrado para {DATASET_NAME}: {DATASET_DIR}")
if len(DATASET_FILES) > 1:
    raise ValueError(
        f"Mais de um Parquet encontrado em {DATASET_DIR}; selecione o arquivo/período explicitamente: "
        f"{[path.name for path in DATASET_FILES]}"
    )
DATASET_PATH = DATASET_FILES[0]
if not DATASET_PATH.stem.startswith(f"{DATASET_NAME}-"):
    raise ValueError(
        f"O nome do arquivo deve começar com '{DATASET_NAME}-' para derivar o período: "
        f"{DATASET_PATH.name}"
    )

PERIOD_LABEL = DATASET_PATH.stem[len(DATASET_NAME) + 1:]
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
ENCODE_BATCH_SIZE = 64
MAX_DESCRIPTIONS = 10_000  # Amostra reprodutível para limitar o custo do experimento.
SAMPLE_SEEDS = [42]
MIN_CLUSTER_SIZE_VALUES = [2, 5, 10, 20]
MIN_SAMPLES_VALUES = [None, 2, 5, 10]
CLUSTER_SELECTION_METHOD = "eom"  # Alternativa: "leaf".
HDBSCAN_N_JOBS = 1
SILHOUETTE_SAMPLE_SIZE = 10_000
OUTPUT_ROOT = Path("reports/mpnet_hdbscan")
MODEL_ROOT = Path("models/mpnet_hdbscan")


TSNE_MAX_POINTS = 5_000  # Amostra fixa para limitar o custo da projeção.
TSNE_PERPLEXITY = 30
TSNE_MAX_ITER = 1_000
TSNE_RANDOM_STATE = 42
TSNE_3D_N_COMPONENTS = 3
TSNE_PCA_COMPONENTS = 50  # Número máximo de componentes PCA antes do t-SNE.


### Configuração do ranking final


In [3]:
# O limite filtra apenas o ranking e a seleção final; não altera os ajustes do HDBSCAN.
MAX_CLUSTERS_FOR_RANKING = 200

# Os pesos definem a contribuição relativa de silhouette e cobertura (1 - ruído).
SILHOUETTE_WEIGHT = 0.5
NOISE_WEIGHT = 0.5


## 2. Funções da análise

A preparação remove descrições vazias, mantém uma linha por descrição e valida que a frequência total não foi alterada. Cada configuração HDBSCAN de uma semente usa os mesmos embeddings MPNet.


In [4]:
def prepare_items(frame):
    """Limpa descrições, agrega duplicatas e preserva a frequência total."""
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = (
        prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY]
        .sum()
    )
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A preparação alterou a soma de frequência.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)

def summarize_clusters(items, labels):
    result = items.copy()
    result["cluster"] = labels
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary

def score_configuration(matrix, labels, sample_size, random_state):
    labels = np.asarray(labels)
    noise_mask = labels == -1
    clustered_positions = np.flatnonzero(~noise_mask)
    clustered_labels = labels[clustered_positions]
    cluster_count = len(np.unique(clustered_labels))
    score = np.nan
    silhouette_count = min(sample_size, len(clustered_labels))
    if silhouette_count:
        rng = np.random.default_rng(random_state)
        sample_positions = np.sort(rng.choice(
            clustered_positions, size=silhouette_count, replace=False
        ))
        sample_labels = labels[sample_positions]
        sample_cluster_count = len(np.unique(sample_labels))
        if 2 <= sample_cluster_count < silhouette_count:
            score = float(silhouette_score(matrix[sample_positions], sample_labels, metric="cosine"))
        else:
            silhouette_count = 0
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_sample_size": silhouette_count,
        "silhouette_cosine_without_noise": score,
    }

def compute_tsne_projection(features, max_points, perplexity, max_iter, random_state, n_components=2):
    """Calcula uma projeção t-SNE reprodutível sobre uma amostra das linhas."""
    point_count = features.shape[0]
    if point_count < 3:
        raise ValueError("O t-SNE requer pelo menos três descrições.")

    sample_count = min(point_count, max_points)
    if sample_count < 3:
        raise ValueError("TSNE_MAX_POINTS deve ser pelo menos tres.")
    rng = np.random.default_rng(random_state)
    indices = np.sort(rng.choice(point_count, size=sample_count, replace=False))
    sampled_features = features[indices]

    if sampled_features.shape[1] > 1:
        component_count = min(TSNE_PCA_COMPONENTS, sampled_features.shape[1], sample_count - 1)
        reduced_features = PCA(
            n_components=component_count, random_state=random_state,
        ).fit_transform(sampled_features)
    else:
        reduced_features = sampled_features

    effective_perplexity = min(float(perplexity), (sample_count - 1) / 3)
    coordinates = TSNE(
        n_components=n_components,
        perplexity=effective_perplexity,
        init="pca",
        learning_rate="auto",
        max_iter=max_iter,
        random_state=random_state,
        method="barnes_hut",
    ).fit_transform(reduced_features)
    return indices, coordinates, reduced_features.shape[1]


def save_tsne_plot(coordinates, labels, run_id, output_path):
    """Salva um PNG t-SNE colorido pelos clusters, com o ruído em cinza."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_ids = np.unique(labels[~noise_mask])
    if cluster_ids.size:
        palette = plt.get_cmap("tab20").colors
        cluster_cmap = ListedColormap([
            palette[index % len(palette)] for index in range(len(cluster_ids))
        ])
        cluster_codes = np.searchsorted(cluster_ids, labels[~noise_mask])
    else:
        cluster_cmap = None
        cluster_codes = np.empty(0, dtype=int)

    figure, axis = plt.subplots(figsize=(11, 8), constrained_layout=True)
    if noise_mask.any():
        axis.scatter(
            coordinates[noise_mask, 0], coordinates[noise_mask, 1],
            c="#b8b8b8", s=8, alpha=0.3, linewidths=0,
        )
    if cluster_ids.size:
        axis.scatter(
            coordinates[~noise_mask, 0], coordinates[~noise_mask, 1],
            c=cluster_codes, cmap=cluster_cmap, vmin=-0.5,
            vmax=len(cluster_ids) - 0.5, s=9, alpha=0.75, linewidths=0,
        )
    axis.set_title(
        f"t-SNE: {run_id}\n{len(cluster_ids)} clusters; "
        f"{int(noise_mask.sum()):,} descrições como ruído"
    )
    axis.set_xlabel("t-SNE 1")
    axis.set_ylabel("t-SNE 2")
    axis.text(
        0.01, 0.01,
        "Ruído em cinza; as cores se repetem a cada 20 clusters.",
        transform=axis.transAxes, fontsize=8, color="#555555",
    )
    figure.savefig(output_path, dpi=160)
    plt.close(figure)


def save_tsne_3d_html(coordinates, labels, run_id, output_path):
    """Salva uma visualização t-SNE 3D interativa e autocontida em HTML."""
    points = [
        {"x": float(x), "y": float(y), "z": float(z), "label": int(label)}
        for (x, y, z), label in zip(coordinates, labels)
    ]
    data_json = json.dumps(points, separators=(",", ":"))
    title = str(run_id).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    page = r"""<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>t-SNE 3D: __TITLE__</title>
<style>
html, body { margin: 0; height: 100%; background: #f7f8fa; color: #202124; font: 14px system-ui, sans-serif; }
main { display: flex; flex-direction: column; height: 100%; }
header { padding: 12px 18px; background: white; border-bottom: 1px solid #ddd; }
h1 { margin: 0 0 4px; font-size: 17px; }
p { margin: 0; color: #555; }
canvas { display: block; width: 100%; flex: 1; min-height: 300px; cursor: grab; }
canvas:active { cursor: grabbing; }
</style>
</head>
<body>
<main>
<header><h1>Projeção t-SNE 3D: __TITLE__</h1><p>Arraste para girar; use a roda do mouse para aproximar. Ruído em cinza.</p></header>
<canvas id="plot"></canvas>
</main>
<script>
const points = __POINTS__;
const canvas = document.getElementById("plot");
const ctx = canvas.getContext("2d");
let yaw = 0.65, pitch = -0.35, zoom = 1, dragging = false, lastX = 0, lastY = 0;
const colors = ["#3366cc", "#dc3912", "#ff9900", "#109618", "#990099", "#0099c6", "#dd4477", "#66aa00", "#b82e2e", "#316395", "#994499", "#22aa99", "#aaaa11", "#6633cc", "#e67300", "#8b0707", "#651067", "#329262", "#5574a6", "#3b3eac"];
const bounds = ["x", "y", "z"].map(key => {
  const values = points.map(point => point[key]);
  const low = Math.min(...values), high = Math.max(...values);
  return { low, span: Math.max(high - low, 1e-9) };
});
function resize() {
  const ratio = window.devicePixelRatio || 1;
  canvas.width = Math.max(1, Math.floor(canvas.clientWidth * ratio));
  canvas.height = Math.max(1, Math.floor(canvas.clientHeight * ratio));
  ctx.setTransform(ratio, 0, 0, ratio, 0, 0);
  draw();
}
function draw() {
  const width = canvas.clientWidth, height = canvas.clientHeight;
  ctx.clearRect(0, 0, width, height);
  const scale = Math.min(width, height) * 0.7 * zoom;
  const cy = Math.cos(yaw), sy = Math.sin(yaw), cp = Math.cos(pitch), sp = Math.sin(pitch);
  const projected = points.map(point => {
    const x = ((point.x - bounds[0].low) / bounds[0].span - 0.5) * 2;
    const y = ((point.y - bounds[1].low) / bounds[1].span - 0.5) * 2;
    const z = ((point.z - bounds[2].low) / bounds[2].span - 0.5) * 2;
    const rx = x * cy - z * sy, rz = x * sy + z * cy;
    const ry = y * cp - rz * sp, depth = y * sp + rz * cp;
    return { x: width / 2 + rx * scale / 2, y: height / 2 - ry * scale / 2, depth, label: point.label };
  }).sort((a, b) => a.depth - b.depth);
  for (const point of projected) {
    ctx.globalAlpha = point.label === -1 ? 0.24 : 0.78;
    ctx.fillStyle = point.label === -1 ? "#999999" : colors[Math.abs(point.label) % colors.length];
    ctx.beginPath();
    ctx.arc(point.x, point.y, point.label === -1 ? 2 : 3, 0, Math.PI * 2);
    ctx.fill();
  }
  ctx.globalAlpha = 1;
}
canvas.addEventListener("pointerdown", event => { dragging = true; lastX = event.clientX; lastY = event.clientY; canvas.setPointerCapture(event.pointerId); });
canvas.addEventListener("pointerup", () => { dragging = false; });
canvas.addEventListener("pointermove", event => {
  if (!dragging) return;
  yaw += (event.clientX - lastX) * 0.008;
  pitch = Math.max(-1.5, Math.min(1.5, pitch + (event.clientY - lastY) * 0.008));
  lastX = event.clientX; lastY = event.clientY; draw();
});
canvas.addEventListener("wheel", event => { event.preventDefault(); zoom = Math.max(0.3, Math.min(5, zoom * (event.deltaY < 0 ? 1.1 : 0.9))); draw(); }, { passive: false });
window.addEventListener("resize", resize);
resize();
</script>
</body>
</html>"""
    page = page.replace("__TITLE__", title).replace("__POINTS__", data_json)
    output_path.write_text(page, encoding="utf-8")


## 3. Carregar e preparar dados

Origem: Parquet produzido pelos notebooks de preparação de dados do projeto, com descrições de supermercado, NCMs e duplicatas agregadas por descrição. A amostragem é feita uma vez por semente. Os embeddings são gerados uma vez por amostra e reutilizados na grade de parâmetros.


In [5]:
raw_items = pd.read_parquet(DATASET_PATH)
required_columns = {DESCRIPTION_NORMALIZED, FREQUENCY}
missing_columns = required_columns - set(raw_items.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")
base_items = prepare_items(raw_items)
logger.info("Descrições disponíveis após preparação: %d", len(base_items))

# Cada execução usa uma pasta nova para preservar relatórios anteriores.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)


2026-10-18 10:55:18,191 INFO Descrições disponíveis após preparação: 149149


## 4. Gerar embeddings, ajustar HDBSCAN e comparar parâmetros

MPNet gera embeddings densos normalizados e a distância cosseno é usada diretamente no HDBSCAN. O ruído (`cluster = -1`) é excluído da silhouette e contabilizado separadamente; configurações com menos de dois clusters válidos ficam sem silhouette. Para controlar o custo, a silhouette usa no máximo `SILHOUETTE_SAMPLE_SIZE` itens não classificados como ruído.

Para visualização, uma amostra fixa de até `TSNE_MAX_POINTS` descrições é reduzida com PCA e projetada uma vez por amostra. As coordenadas são reutilizadas para salvar um PNG por configuração.

A projeção t-SNE 3D é calculada separadamente e cada configuração também é salva como HTML interativo autocontido em `tsne_3d_html/`.


In [6]:
records = []
labels_by_run = {}
items_by_run = {}
matrices_by_representation = {}
items_by_seed = {}
vectors_by_seed = {}

logger.info("Carregando modelo %s", MODEL_NAME)
encoder = SentenceTransformer(MODEL_NAME)

for sample_seed in SAMPLE_SEEDS:
    sampled_items = base_items
    if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
        sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=sample_seed)
    sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    items_by_seed[sample_seed] = sampled_items

    logger.info("Gerando embeddings MPNet: seed=%d, descrições=%d", sample_seed, len(sampled_items))
    vectors = encoder.encode(
        sampled_items[DESCRIPTION_NORMALIZED].astype(str).tolist(),
        batch_size=ENCODE_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)
    vectors_by_seed[sample_seed] = vectors
    matrices_by_representation[(sample_seed, "mpnet")] = vectors

    for min_cluster_size in MIN_CLUSTER_SIZE_VALUES:
        for min_samples in MIN_SAMPLES_VALUES:
            min_samples_label = "default" if min_samples is None else str(min_samples)
            run_id = (
                f"seed{sample_seed}_mpnet_min_cluster_size{min_cluster_size}"
                f"_min_samples{min_samples_label}_{CLUSTER_SELECTION_METHOD}"
            )
            logger.info("Ajustando HDBSCAN %s", run_id)
            model = HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric="cosine",
                algorithm="brute",
                cluster_selection_method=CLUSTER_SELECTION_METHOD,
                n_jobs=HDBSCAN_N_JOBS,
                copy=True,
            )
            labels = model.fit_predict(vectors)
            metrics = score_configuration(
                vectors, labels, SILHOUETTE_SAMPLE_SIZE,
                TSNE_RANDOM_STATE + sample_seed,
            )
            records.append({
                "run_id": run_id, "sample_seed": sample_seed,
                "representation": "mpnet", "model_name": MODEL_NAME,
                "min_cluster_size": min_cluster_size,
                "min_samples": min_samples,
                "cluster_selection_method": CLUSTER_SELECTION_METHOD,
                "frequency_weighted": False, **metrics,
            })
            labels_by_run[run_id] = labels
            items_by_run[run_id] = sampled_items

results = pd.DataFrame(records)


2026-10-18 10:55:18,243 INFO Carregando modelo sentence-transformers/paraphrase-multilingual-mpnet-base-v2
2026-10-18 10:55:18,248 INFO No device provided, using cpu
2026-10-18 10:55:18,582 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-18 10:55:18,584 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-18 10:55:18,610 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/modules.json?%2Fsentence-transformers%2Fparaphrase-multilingual-mpnet-base-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22f7640f94e81bb7f4f04daf1668850b38763a13d9%22 "HTTP/1.1 200 OK"
2026-10-18 10:55:18,761 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/pa

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-10-18 10:55:22,548 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
2026-10-18 10:55:22,725 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-18 10:55:22,864 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-18 10:55:26,857 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
2026-10-18 10:55:27,030 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-18 10:55:27,059 INFO HTTP Request: HEAD https:

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2026-10-18 11:00:53,929 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size2_min_samplesdefault_eom
2026-10-18 11:01:01,872 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size2_min_samples2_eom
2026-10-18 11:01:09,220 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size2_min_samples5_eom
2026-10-18 11:01:15,179 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size2_min_samples10_eom
2026-10-18 11:01:20,943 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size5_min_samplesdefault_eom
2026-10-18 11:01:26,643 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size5_min_samples2_eom
2026-10-18 11:01:33,007 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size5_min_samples5_eom
2026-10-18 11:01:39,587 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size5_min_samples10_eom
2026-10-18 11:01:47,930 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size10_min_samplesdefault_eom
2026-10-18 11:01:56,314 INFO Ajustando HDBSCAN seed42_mpnet_min_cluster_size10_min_samples2_eom
2026-10-18 11:02:02,538 INFO

### Calcular a metrica composta e ordenar os resultados

A silhouette e reescalada para 0 a 1 e combinada com a proporcao de descricoes classificadas (`1 - noise_fraction`). O score composto determina a ordem; silhouette e ruido servem apenas de desempate quando os scores forem iguais.


In [7]:
if SILHOUETTE_WEIGHT < 0 or NOISE_WEIGHT < 0 or SILHOUETTE_WEIGHT + NOISE_WEIGHT <= 0:
    raise ValueError("Os pesos devem ser nao negativos e pelo menos um deles deve ser maior que zero.")

# Silhouette varia de -1 a 1; reescale para 0 a 1 para combinar com cobertura.
results["silhouette_quality"] = (results["silhouette_cosine_without_noise"] + 1) / 2
results["noise_quality"] = 1 - results["noise_fraction"]
weight_total = SILHOUETTE_WEIGHT + NOISE_WEIGHT
results["composite_score"] = (
    SILHOUETTE_WEIGHT * results["silhouette_quality"]
    + NOISE_WEIGHT * results["noise_quality"]
) / weight_total

# O score composto e o criterio principal; os outros criterios so desempataram scores iguais.
ranking = (
    results.loc[
        results["cluster_count_excluding_noise"].le(MAX_CLUSTERS_FOR_RANKING)
    ]
    .sort_values(
        ["composite_score", "silhouette_cosine_without_noise", "noise_fraction"],
        ascending=[False, False, True], na_position="last",
    )
    .reset_index(drop=True)
)
if ranking.empty:
    raise ValueError(
        "Nenhum resultado atende ao limite de "
        f"{MAX_CLUSTERS_FOR_RANKING} clusters. Ajuste min_cluster_size/min_samples ou o limite."
    )
# Explicita a posicao de cada configuracao depois dos filtros e criterios acima.
ranking.insert(0, "rank", range(1, len(ranking) + 1))


In [8]:
display(ranking)

,rank,run_id,sample_seed,representation,model_name,min_cluster_size,min_samples,cluster_selection_method,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_sample_size,silhouette_cosine_without_noise,silhouette_quality,noise_quality,composite_score
0,1,seed42_mpnet_min_cluster_size10_min_samplesdefault_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,10,NaN,eom,False,3,2164,0.2164,7836,0.128221,0.564110,0.7836,0.673855
1,2,seed42_mpnet_min_cluster_size10_min_samples10_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,10,10.0,eom,False,3,2164,0.2164,7836,0.128221,0.564110,0.7836,0.673855
2,3,seed42_mpnet_min_cluster_size10_min_samples2_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,10,2.0,eom,False,134,6812,0.6812,3188,0.319797,0.659899,0.3188,0.489349
3,4,seed42_mpnet_min_cluster_size5_min_samplesdefault_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,5,NaN,eom,False,144,7844,0.7844,2156,0.422148,0.711074,0.2156,0.463337
4,5,seed42_mpnet_min_cluster_size5_min_samples5_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,5,5.0,eom,False,144,7844,0.7844,2156,0.422148,0.711074,0.2156,0.463337
5,6,seed42_mpnet_min_cluster_size20_min_samples2_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,20,2.0,eom,False,54,7222,0.7222,2778,0.274496,0.637248,0.2778,0.457524
6,7,seed42_mpnet_min_cluster_size20_min_samplesdefault_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,20,NaN,eom,False,2,6538,0.6538,3462,0.121596,0.560798,0.3462,0.453499
7,8,seed42_mpnet_min_cluster_size10_min_samples5_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,10,5.0,eom,False,75,7926,0.7926,2074,0.392496,0.696248,0.2074,0.451824
8,9,seed42_mpnet_min_cluster_size5_min_samples10_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,5,10.0,eom,False,67,8452,0.8452,1548,0.452500,0.726250,0.1548,0.440525
9,10,seed42_mpnet_min_cluster_size20_min_samples5_eom,42,mpnet,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,20,5.0,eom,False,33,8252,0.8252,1748,0.369017,0.684508,0.1748,0.429654


In [9]:
ranking = (
    results.loc[
        results["cluster_count_excluding_noise"].le(MAX_CLUSTERS_FOR_RANKING)
        & results["silhouette_cosine_without_noise"].notna()
        & results["silhouette_cosine_without_noise"].gt(results["noise_fraction"])
    ]
    .sort_values(
        ["composite_score", "silhouette_cosine_without_noise", "noise_fraction"],
        ascending=[False, False, True],
        na_position="last",
    )
    .reset_index(drop=True)
)

ranking.head(40)

,run_id,sample_seed,representation,model_name,min_cluster_size,min_samples,cluster_selection_method,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_sample_size,silhouette_cosine_without_noise,silhouette_quality,noise_quality,composite_score


## Salvar assignments de um teste específico


In [12]:
ASSIGNMENT_RUN_ID = "seed42_mpnet_min_cluster_size10_min_samples2_eom"

run_assignments, run_cluster_summary = summarize_clusters(
    items_by_run[ASSIGNMENT_RUN_ID],
    labels_by_run[ASSIGNMENT_RUN_ID],
)

assignment_path = output_dir / f"assignments_{ASSIGNMENT_RUN_ID}.csv"
if assignment_path.exists():
    raise FileExistsError(f"Arquivo já existe: {assignment_path}")

run_assignments.to_csv(assignment_path, index=False, encoding="utf-8-sig")
print(f"Assignments salvos em: {assignment_path}")


Assignments salvos em: reports\mpnet_hdbscan\202501-202512_20261018T135518Z\assignments_seed42_mpnet_min_cluster_size10_min_samples2_eom.csv


## 5. Revisar um resultado e salvar relatórios

Escolha `SELECTED_RUN_ID` após comparar quantidade de clusters, ruído e silhouette. A tabela de resumo inclui `-1` como grupo de ruído. Os arquivos são gravados em uma pasta com identificador UTC único para preservar execuções anteriores.

Os gráficos ficam em `tsne_plots/`, um PNG por configuração, e as coordenadas da amostra em `tsne_coordinates/`, dentro da pasta da execução.


In [15]:
SELECTED_RUN_ID = ASSIGNMENT_RUN_ID
if SELECTED_RUN_ID not in results["run_id"].values:
    raise ValueError(f"Execucao nao encontrada em results: {SELECTED_RUN_ID}")
if SELECTED_RUN_ID not in items_by_run or SELECTED_RUN_ID not in labels_by_run:
    raise ValueError(f"Execucao sem itens ou labels carregados: {SELECTED_RUN_ID}")
selected = results.set_index("run_id").loc[SELECTED_RUN_ID]
selected_items = items_by_run[SELECTED_RUN_ID]
selected_labels = labels_by_run[SELECTED_RUN_ID]
assignments, cluster_summary = summarize_clusters(selected_items, selected_labels)
cluster_summary = cluster_summary.sort_values("frequencia_total", ascending=False).reset_index(drop=True)
cluster_sizes = assignments.loc[assignments["cluster"].ne(-1)].groupby("cluster")[DESCRIPTION_NORMALIZED].size()
singleton_clusters = cluster_sizes[cluster_sizes.eq(1)].index
frequent_singletons = assignments[assignments["cluster"].isin(singleton_clusters)].sort_values(
    FREQUENCY, ascending=False
)
frequent_noise = assignments[assignments["cluster"].eq(-1)].sort_values(
    FREQUENCY, ascending=False
)
display(cluster_summary.head(20))
display(frequent_singletons.head(20))
display(frequent_noise.head(20))

results.to_csv(output_dir / "metrics.csv", index=False)
assignments.to_csv(output_dir / "assignments.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(output_dir / "cluster_summary.csv", index=False, encoding="utf-8-sig")
frequent_singletons.to_csv(
    output_dir / "frequent_singleton_descriptions.csv", index=False, encoding="utf-8-sig"
)
frequent_noise.to_csv(
    output_dir / "frequent_noise_descriptions.csv", index=False, encoding="utf-8-sig"
)

,cluster,descricoes_unicas,frequencia_total
0,-1,6812,31941
1,65,24,2053
2,75,31,1452
3,57,62,1241
4,32,75,724
5,102,45,584
6,72,124,572
7,124,102,536
8,120,34,499
9,109,26,415


,descricao_normalizada,frequencia,cluster,ruido


,descricao_normalizada,frequencia,cluster,ruido
9049,salsicha kg,906,-1,True
6999,paninho de limpeza 28x38,614,-1,True
5500,lanche,593,-1,True
2846,cola liquida branca go office 90g,514,-1,True
6491,molho ingles,463,-1,True
281,acucar cristal acond p eantrib,395,-1,True
2440,catchup,394,-1,True
4911,goiabada,381,-1,True
280,acucar cristal acond f eantrib,344,-1,True
5675,linguica,256,-1,True


## Salvar um modelo pelo Run ID

Informe o `MODEL_RUN_ID` abaixo para exportar o encoder MPNet, o HDBSCAN ajustado, suas configurações e os assignments de treino. A exportação cria uma pasta nova em `models/mpnet_hdbscan/` para preservar modelos anteriores. O artefato HDBSCAN não fornece atribuição direta de novos pontos a clusters; o encoder é registrado pelo identificador do modelo usado.


In [16]:
MODEL_RUN_ID = "seed42_mpnet_min_cluster_size10_min_samples2_eom"

if MODEL_RUN_ID not in labels_by_run:
    raise ValueError(f"Run ID não encontrado. Opções: {list(labels_by_run)}")

model_run = results.set_index("run_id").loc[MODEL_RUN_ID]
model_items = items_by_run[MODEL_RUN_ID]
model_labels = labels_by_run[MODEL_RUN_ID]
model_vectors = encoder.encode(
    model_items[DESCRIPTION_NORMALIZED].astype(str).tolist(),
    batch_size=ENCODE_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)
model_min_samples = (
    None if pd.isna(model_run["min_samples"]) else int(model_run["min_samples"])
)
saved_model = HDBSCAN(
    min_cluster_size=int(model_run["min_cluster_size"]),
    min_samples=model_min_samples,
    metric="cosine",
    algorithm="brute",
    cluster_selection_method=model_run["cluster_selection_method"],
    n_jobs=HDBSCAN_N_JOBS,
    copy=True,
)
saved_model.fit(model_vectors)
if not np.array_equal(saved_model.labels_, model_labels):
    raise RuntimeError("Os labels do modelo refeito não coincidem com os assignments do Run ID.")

model_assignments, _ = summarize_clusters(model_items, model_labels)
model_output_dir = MODEL_ROOT / f"{PERIOD_LABEL}_{RUN_ID}_{MODEL_RUN_ID}"
model_output_dir.mkdir(parents=True, exist_ok=False)
encoder.save(str(model_output_dir / "sentence_transformer"))
pd.to_pickle(saved_model, model_output_dir / "hdbscan_model.pkl")
model_assignments.to_csv(
    model_output_dir / "training_assignments.csv", index=False, encoding="utf-8-sig"
)
model_config = {
    "run_id": MODEL_RUN_ID,
    "period": PERIOD_LABEL,
    "dataset_name": DATASET_NAME,
    "sample_seed": int(model_run["sample_seed"]),
    "representation": "mpnet",
    "encoder": {
        "model_name": MODEL_NAME,
        "encode_batch_size": ENCODE_BATCH_SIZE,
        "normalize_embeddings": True,
        "embedding_dimension": int(model_vectors.shape[1]),
    },
    "hdbscan": {
        "min_cluster_size": int(model_run["min_cluster_size"]),
        "min_samples": model_min_samples,
        "metric": "cosine",
        "algorithm": "brute",
        "cluster_selection_method": model_run["cluster_selection_method"],
    },
    "frequency_weighted": False,
    "artifacts": {
        "encoder": "sentence_transformer/",
        "model": "hdbscan_model.pkl",
        "training_assignments": "training_assignments.csv",
    },
}
(model_output_dir / "config.json").write_text(
    json.dumps(model_config, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Modelo do Run ID %s salvo em %s", MODEL_RUN_ID, model_output_dir)


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2026-10-18 11:13:34,087 INFO Saving model to models\mpnet_hdbscan\202501-202512_20261018T135518Z_seed42_mpnet_min_cluster_size10_min_samples2_eom\sentence_transformer


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-18 11:13:50,645 INFO Modelo do Run ID seed42_mpnet_min_cluster_size10_min_samples2_eom salvo em models\mpnet_hdbscan\202501-202512_20261018T135518Z_seed42_mpnet_min_cluster_size10_min_samples2_eom


## 6. Gerar visualizações t-SNE

Esta etapa roda depois do ajuste e do ranking. Ela projeta os embeddings MPNet já calculados e salva as coordenadas e os gráficos, sem executar novamente o HDBSCAN.


In [17]:
# Calcula projeçãoções 2D e 3D separadas para cada amostra MPNet.
tsne_by_representation = {}
tsne_3d_by_representation = {}
for (sample_seed, representation), matrix in matrices_by_representation.items():
    logger.info("Calculando t-SNE 2D para %s, seed=%d", representation, sample_seed)
    tsne_indices, tsne_coordinates, tsne_dimensions = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE + sample_seed,
    )
    tsne_by_representation[(sample_seed, representation)] = {
        "indices": tsne_indices, "coordinates": tsne_coordinates,
        "reduced_dimensions": tsne_dimensions,
    }

    logger.info("Calculando t-SNE 3D para %s, seed=%d", representation, sample_seed)
    tsne_3d_indices, tsne_3d_coordinates, tsne_3d_dimensions = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE + sample_seed, n_components=TSNE_3D_N_COMPONENTS,
    )
    if not np.array_equal(tsne_indices, tsne_3d_indices):
        raise ValueError("As projeçãoções 2D e 3D devem usar a mesma amostra de descrições.")
    tsne_3d_by_representation[(sample_seed, representation)] = {
        "indices": tsne_3d_indices, "coordinates": tsne_3d_coordinates,
        "reduced_dimensions": tsne_3d_dimensions,
    }

# Salva coordenadas 2D e um PNG por configuração.
tsne_coordinates_dir = output_dir / "tsne_coordinates"
tsne_plots_dir = output_dir / "tsne_plots"
tsne_3d_html_dir = output_dir / "tsne_3d_html"
tsne_coordinates_dir.mkdir()
tsne_plots_dir.mkdir()
tsne_3d_html_dir.mkdir()
tsne_artifacts = {}
for (sample_seed, representation), projection in tsne_by_representation.items():
    indices = projection["indices"]
    coordinates = projection["coordinates"]
    coordinate_frame = items_by_seed[sample_seed].iloc[indices][
        [DESCRIPTION_NORMALIZED, FREQUENCY]
    ].reset_index(drop=True)
    coordinate_frame["tsne_1"] = coordinates[:, 0]
    coordinate_frame["tsne_2"] = coordinates[:, 1]
    coordinate_name = f"seed{sample_seed}_{representation}_coordinates.csv"
    coordinate_frame.to_csv(
        tsne_coordinates_dir / coordinate_name, index=False, encoding="utf-8-sig",
    )
    tsne_artifacts[f"seed{sample_seed}_{representation}"] = {
        "coordinates_file": f"tsne_coordinates/{coordinate_name}",
        "sample_size": len(indices),
        "reduced_dimensions": projection["reduced_dimensions"],
    }

for run_id, labels in labels_by_run.items():
    result_row = results.loc[results["run_id"].eq(run_id)].iloc[0]
    representation_key = (int(result_row["sample_seed"]), result_row["representation"])
    projection = tsne_by_representation[representation_key]
    projection_3d = tsne_3d_by_representation[representation_key]
    plot_labels = np.asarray(labels)[projection["indices"]]
    plot_path = tsne_plots_dir / f"{run_id}.png"
    save_tsne_plot(projection["coordinates"], plot_labels, run_id, plot_path)
    tsne_artifacts[f"plot:{run_id}"] = str(plot_path.relative_to(output_dir))

    plot_3d_path = tsne_3d_html_dir / f"{run_id}.html"
    save_tsne_3d_html(projection_3d["coordinates"], plot_labels, run_id, plot_3d_path)
    tsne_artifacts[f"plot_3d:{run_id}"] = str(plot_3d_path.relative_to(output_dir))

logger.info("Imagens t-SNE 2D e gráficos HTML 3D salvos: %d", len(labels_by_run))

run_metadata = {
    "dataset_name": DATASET_NAME, "period": PERIOD_LABEL,
    "frequency_weighted": False,
    "dataset_path": str(DATASET_PATH.relative_to(PROJECT_ROOT)), "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seeds": SAMPLE_SEEDS, "model_name": MODEL_NAME,
    "encode_batch_size": ENCODE_BATCH_SIZE,
    "selected_run_id": SELECTED_RUN_ID,
    "min_cluster_size_values": MIN_CLUSTER_SIZE_VALUES,
    "min_samples_values": MIN_SAMPLES_VALUES,
    "cluster_selection_method": CLUSTER_SELECTION_METHOD,
    "silhouette_sample_size": SILHOUETTE_SAMPLE_SIZE,
    "tsne": {
        "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
        "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
        "pca_components": TSNE_PCA_COMPONENTS,
        "three_dimensional_components": TSNE_3D_N_COMPONENTS,
        "artifacts": tsne_artifacts,
    },
}
(output_dir / "run.json").write_text(
    json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Relatórios e imagens t-SNE salvos em %s", output_dir)


2026-10-18 11:13:51,263 INFO Calculando t-SNE 2D para mpnet, seed=42
2026-10-18 11:14:26,861 INFO Calculando t-SNE 3D para mpnet, seed=42
2026-10-18 11:16:08,468 INFO Imagens t-SNE 2D e gráficos HTML 3D salvos: 16
2026-10-18 11:16:08,471 INFO Relatórios e imagens t-SNE salvos em reports\mpnet_hdbscan\202501-202512_20261018T135518Z


## 7. Interpretação

- `noise_fraction` alta significa que muitas descrições não entraram em grupos densos para os parâmetros escolhidos.
- `min_cluster_size` controla o menor grupo selecionável. `min_samples` maior tende a exigir evidência local mais forte e pode aumentar o ruído.
- Silhouette mais alta deve ser lida junto da fração de ruído e do tamanho da amostra usada para calculá-la; uma pontuação calculada em poucos itens pode ser enganosa.
- HDBSCAN não atribui automaticamente novas descrições a clusters neste fluxo e não define categorias sem revisão humana.
- MPNet representa similaridade semântica; isso pode agrupar descrições com vocabulário diferente, mas não substitui a revisão humana nem garante coerência fiscal.

Os HTML em `tsne_3d_html/` contêm uma projeção t-SNE 3D que pode ser girada com o mouse e aberta sem dependências externas.
